In [8]:
from pathlib import Path
import muon as mu
import pandas as pd
import scipy.sparse as sp

In [13]:
tissue_name = "mESC"
sample_name = "E7.5_rep1"

DATA_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data")
h5mu_file = DATA_DIR / "processed" / tissue_name / sample_name / "multiome_processed.h5mu"

mdata = mu.read(h5mu_file)

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [15]:
multigrntools_raw_data_dir = Path(
    "/gpfs/Labs/Uzun/DATA/PROJECTS/"
    "2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS"
)

aligned_mdata = mdata.copy()
mu.pp.intersect_obs(aligned_mdata)

rna_all = aligned_mdata.mod["rna"]
celltype_counts = rna_all.obs["celltype"].value_counts()


# Same mapping-file location and logic as train_tf_to_tg_celltype_model.py.
cell_type_specific_gt_dir = (
    DATA_DIR / "ground_truth_files" / "cell_type_specific"
)
label_map_path = (
    cell_type_specific_gt_dir / f"{tissue_name}_label_map.tsv"
)

if label_map_path.exists():
    label_map = pd.read_csv(
        label_map_path,
        sep="\t",
        comment="#",
    )
    slice_members = (
        label_map.groupby("slice")["paper_celltype"]
        .apply(list)
        .to_dict()
    )
    print(
        f"{label_map_path.name}: {len(slice_members)} slices from "
        f"{label_map['paper_celltype'].nunique()} annotated cell types"
    )
else:
    slice_members = {
        celltype: [celltype]
        for celltype in celltype_counts.index
    }
    print(
        f"No {label_map_path.name}; one slice per annotated cell type "
        f"({len(slice_members)})"
    )

for slice_name, members in slice_members.items():
    # Mapping files may contain cell types that are absent from this sample.
    present_members = [
        celltype
        for celltype in members
        if celltype in celltype_counts.index
    ]

    if not present_members:
        print(
            f"Skipping {slice_name}: none of its mapped cell types "
            f"occur in {sample_name}"
        )
        continue

    # Combine all cells assigned to this mapped slice/lineage.
    slice_mask = (
        rna_all.obs["celltype"]
        .isin(present_members)
        .fillna(False)
        .to_numpy()
    )
    slice_mdata = aligned_mdata[slice_mask].copy()

    rna = slice_mdata.mod["rna"]
    atac = slice_mdata.mod["atac"]

    if not rna.obs_names.equals(atac.obs_names):
        raise ValueError(
            f"RNA and ATAC cells are not aligned for {slice_name!r}"
        )

    output_dir = (
        multigrntools_raw_data_dir
        / tissue_name
        / sample_name
        / slice_name
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    rna_csv_file = output_dir / f"{slice_name}_RNA.csv"
    atac_csv_file = output_dir / f"{slice_name}_ATAC.csv"

    rna_counts = rna.layers["counts"]
    atac_counts = atac.layers["counts"]

    rna_gene_by_cell_counts = pd.DataFrame(
        rna_counts.toarray()
        if sp.issparse(rna_counts)
        else rna_counts,
        index=rna.obs_names,
        columns=rna.var_names,
    ).T

    atac_peak_by_cell_counts = pd.DataFrame(
        atac_counts.toarray()
        if sp.issparse(atac_counts)
        else atac_counts,
        index=atac.obs_names,
        columns=atac.var_names,
    ).T

    print(
        f"\nProcessing {tissue_name} / {sample_name} / {slice_name}: "
        f"{rna.n_obs:,} cells, {rna.n_vars:,} genes, "
        f"{atac.n_vars:,} peaks"
    )
    print(f"  Aggregated cell types: {present_members}")

    print(f"Saving RNA data to {rna_csv_file}")
    rna_gene_by_cell_counts.to_csv(rna_csv_file)

    print(f"Saving ATAC data to {atac_csv_file}")
    atac_peak_by_cell_counts.to_csv(atac_csv_file)

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:

mESC_label_map.tsv: 21 slices from 37 annotated cell types


/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)



Processing mESC / E7.5_rep1 / Blood_progenitors: 159 cells, 18,637 genes, 196,915 peaks
  Aggregated cell types: ['Blood_progenitors_1', 'Blood_progenitors_2']
Saving RNA data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/mESC/E7.5_rep1/Blood_progenitors/Blood_progenitors_RNA.csv
Saving ATAC data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/mESC/E7.5_rep1/Blood_progenitors/Blood_progenitors_ATAC.csv


/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


...


Processing mESC / E7.5_rep1 / PGC: 13 cells, 18,637 genes, 196,915 peaks
  Aggregated cell types: ['PGC']
Saving RNA data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/mESC/E7.5_rep1/PGC/PGC_RNA.csv
Saving ATAC data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/mESC/E7.5_rep1/PGC/PGC_ATAC.csv


/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)



Processing mESC / E7.5_rep1 / Spinal_cord: 19 cells, 18,637 genes, 196,915 peaks
  Aggregated cell types: ['Spinal_cord']
Saving RNA data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/mESC/E7.5_rep1/Spinal_cord/Spinal_cord_RNA.csv
Saving ATAC data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/mESC/E7.5_rep1/Spinal_cord/Spinal_cord_ATAC.csv


/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)



Processing mESC / E7.5_rep1 / Surface_ectoderm: 238 cells, 18,637 genes, 196,915 peaks
  Aggregated cell types: ['Surface_ectoderm']
Saving RNA data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/mESC/E7.5_rep1/Surface_ectoderm/Surface_ectoderm_RNA.csv
Saving ATAC data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/mESC/E7.5_rep1/Surface_ectoderm/Surface_ectoderm_ATAC.csv
